# Load Dataset

Loads Massimo's `large_joined_sample.csv` (the shared 1500-paper corpus), extracts `paperId` from each
paper's Semantic Scholar URL, and saves two files used by `gnn-embeddings.ipynb`:

- `papers.parquet` — clean node table with columns `[idx, paperId, corpusid, title, abstract, year, citationcount, url]`
- `idx_to_paperid.json` — integer index → paperId mapping for submission generation

**Run this once before `gnn-embeddings.ipynb`.**

In [3]:
import re
import json
import pandas as pd
from pathlib import Path

CSV_PATH = Path("large_joined_sample.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {CSV_PATH}. "
        "Run ss-build-dataset.ipynb first to generate large_joined_sample.csv."
    )

df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

Loaded: (1500, 10)
Columns: ['corpusid', 'title', 'abstract', 'year', 'venue', 'authors', 's2fieldsofstudy', 'citationcount', 'referencecount', 'url']


In [4]:
# -------------------------------------------------------
# Extract paperId from Semantic Scholar URL
# URLs look like: https://www.semanticscholar.org/paper/<slug>/<40-char-hex>
# This is the same regex Massimo uses in seg-embeddings.ipynb
# -------------------------------------------------------

def extract_paper_id(url: str) -> str | None:
    url = str(url).strip()
    if not url:
        return None
    m = re.search(r"/paper/(?:[^/]+/)?([0-9a-f]{40})$", url)
    if m:
        return m.group(1)
    last = url.rstrip("/").split("/")[-1]
    if re.fullmatch(r"[0-9a-f]{40}", last):
        return last
    return None

df["paperId"] = df["url"].apply(extract_paper_id)

total     = len(df)
with_id   = df["paperId"].notna().sum()
missing   = df["paperId"].isna().sum()

print(f"Total papers  : {total}")
print(f"Valid paperId : {with_id}")
print(f"Missing       : {missing}")

if missing > 0:
    print("\nSample rows with missing paperId:")
    print(df[df["paperId"].isna()][["corpusid", "url"]].head(5))

Total papers  : 1500
Valid paperId : 1500
Missing       : 0


In [5]:
# -------------------------------------------------------
# Build clean node table
# Drop rows with no paperId — they cannot appear in submissions
# Reset index so idx is a clean 0-based integer
# -------------------------------------------------------

KEEP_COLS = ["corpusid", "title", "abstract", "year", "citationcount", "url", "paperId"]

papers = (
    df[KEEP_COLS]
    .copy()
    .dropna(subset=["paperId"])
    .reset_index(drop=True)
)
papers.index.name = "idx"
papers = papers.reset_index()

# normalise text columns
papers["title"]    = papers["title"].fillna("").astype(str)
papers["abstract"] = papers["abstract"].fillna("").astype(str)
papers["corpusid"] = papers["corpusid"].astype(str)

print("Node table shape:", papers.shape)
print(papers[["idx", "paperId", "corpusid", "title"]].head(3))

Node table shape: (1500, 8)
   idx                                   paperId   corpusid  \
0    0  747feadb4b33b196ea63c800319bdc5c114fabe8   15884564   
1    1  1ec8cd074b53277f1fbca13698757c54f56e848d  251335420   
2    2  52ecf856e5ea4038217186d284b2a09afac7e726  238991506   

                                               title  
0  C-Terminal Amino Acids 471-507 of Avian Hepati...  
1  Lethality of Police Shootings and Proximity to...  
2  Real world data on symptomology and diagnostic...  


In [ ]:
# -------------------------------------------------------
# Save outputs
# -------------------------------------------------------

papers.to_parquet("papers.parquet", index=False, engine="fastparquet")
print("Saved papers.parquet")

idx_to_paperid = dict(zip(papers["idx"].tolist(), papers["paperId"].tolist()))
with open("idx_to_paperid.json", "w", encoding="utf-8") as f:
    json.dump(idx_to_paperid, f)
print("Saved idx_to_paperid.json")

print(f"\nReady: {len(papers)} papers with valid paperIds")

ArrowKeyError: No type extension with name arrow.py_extension_type found